# Case Study: OEM1 Emissions Investigation

## Objective

The objective of this case study is to identify all OEM1 vehicles affected by the potentially defective T2 control units and determine the municipality in which these vehicles were registered.

According to the investigation information, affected T2 control units were produced by manufacturer 202 in plant 2022 between April 2009 and November 2014.

In addition, control units produced by manufacturer 201 in plant 2011 are affected when their production numbers range from 1250 to 19500.

The affected T2 control units are installed in OEM1 engines. These engines can be installed in OEM1 Type11 and Type12 vehicles.

The analysis therefore follows the supply chain:

**T2 control units → K1 engine components → OEM1 Type11/Type12 vehicles → vehicle registrations → municipality**

The available data are inspected and combined to create the final set of affected registered vehicles.

## 1. Data Selection

The available database contains information about individual parts, components, vehicles, registrations, geodata and logistics.

For this case study, the relevant datasets are:

- `Einzelteil_T02.txt` – production data for T2 control units
- `Bestandteile_Komponente_K1BE1.csv`
- `Bestandteile_Komponente_K1BE2.csv`
- `Bestandteile_Komponente_K1DI1.csv`
- `Bestandteile_Komponente_K1DI2.csv`
- `Bestandteile_Fahrzeuge_OEM1_Typ11.csv`
- `Bestandteile_Fahrzeuge_OEM1_Typ12.csv`
- `Zulassungen_alle_Fahrzeuge.csv`

The K1 component tables are required because they contain the relationship between T2 control units and the K1 engine components.

The OEM1 vehicle tables are required because they contain the relationship between the engine component and the vehicle.

Finally, the registration table is required to determine the municipality in which an affected vehicle was registered.

In [1]:
import pandas as pd

## 2. Import and Prepare the T2 Data

The original `Einzelteil_T02.txt` file contains records separated by tab characters.

The checkpoint analysis showed that the file contains two sets of T2 records. To make the file readable as a normal table, the tab characters are replaced by line breaks and the resulting file is read using whitespace separation.

The resulting dataset contains the T2 identifier, production date, manufacturer, production plant and defect information.

In [2]:
input_file = r".\data\Einzelteil_T02.txt"
fixed_file = r".\data\Einzelteil_T02_fixed.txt"

with open(input_file, "r", encoding="utf-8") as src, \
     open(fixed_file, "w", encoding="utf-8") as dst:

    while chunk := src.read(10_000_000):
        dst.write(chunk.replace("\t", "\n"))

df_t02 = pd.read_csv(
    fixed_file,
    sep=r"\s+",
    quotechar='"',
    na_values="NA"
)

print("Shape:", df_t02.shape)
print(df_t02.columns.tolist())
display(df_t02.head())

C:\Users\Usama Kaleem\AppData\Local\Temp\ipykernel_26480\1368112476.py:10: DtypeWarning: Columns (0: Produktionsdatum.x, 1: Herstellernummer.x, 2: Fehlerhaft_Fahrleistung.x, 3: Produktionsdatum.y, 4: Herstellernummer.y, 5: Fehlerhaft_Fahrleistung.y) have mixed types. Specify dtype option on import or set low_memory=False.
  df_t02 = pd.read_csv(


Shape: (3204104, 15)
['X1', 'ID_T02.x', 'Produktionsdatum.x', 'Herstellernummer.x', 'Werksnummer.x', 'Fehlerhaft.x', 'Fehlerhaft_Datum.x', 'Fehlerhaft_Fahrleistung.x', 'ID_T02.y', 'Produktionsdatum.y', 'Herstellernummer.y', 'Werksnummer.y', 'Fehlerhaft.y', 'Fehlerhaft_Datum.y', 'Fehlerhaft_Fahrleistung.y']


,X1,ID_T02.x,Produktionsdatum.x,Herstellernummer.x,Werksnummer.x,Fehlerhaft.x,Fehlerhaft_Datum.x,Fehlerhaft_Fahrleistung.x,ID_T02.y,Produktionsdatum.y,Herstellernummer.y,Werksnummer.y,Fehlerhaft.y,Fehlerhaft_Datum.y,Fehlerhaft_Fahrleistung.y
1,4,2-201-2011-239,2008-11-07,201.0,2011.0,1.0,2010-04-09,38354.158904,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5,2-201-2011-304,2008-11-07,201.0,2011.0,0.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9,2-201-2011-125,2008-11-07,201.0,2011.0,1.0,2010-04-09,38354.158904,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,17,2-201-2011-55,2008-11-07,201.0,2011.0,0.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,19,2-201-2011-133,2008-11-07,201.0,2011.0,0.0,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Create a Unified T2 Dataset

The original T2 data contain two sets of columns (`.x` and `.y`).

The two sets are combined using `combine_first()` so that the final dataset contains one consistent set of T2 attributes.

This produces one row per T2 control unit with:

- T2 ID
- production date
- manufacturer
- production plant
- defect information
- defect date
- mileage at defect

In [3]:
t02 = pd.DataFrame({
    "ID_T02": df_t02["ID_T02.x"].combine_first(df_t02["ID_T02.y"]),
    "Produktionsdatum": df_t02["Produktionsdatum.x"].combine_first(
        df_t02["Produktionsdatum.y"]
    ),
    "Herstellernummer": df_t02["Herstellernummer.x"].combine_first(
        df_t02["Herstellernummer.y"]
    ),
    "Werksnummer": df_t02["Werksnummer.x"].combine_first(
        df_t02["Werksnummer.y"]
    ),
    "Fehlerhaft": df_t02["Fehlerhaft.x"].combine_first(
        df_t02["Fehlerhaft.y"]
    ),
    "Fehlerhaft_Datum": df_t02["Fehlerhaft_Datum.x"].combine_first(
        df_t02["Fehlerhaft_Datum.y"]
    ),
    "Fehlerhaft_Fahrleistung": df_t02[
        "Fehlerhaft_Fahrleistung.x"
    ].combine_first(
        df_t02["Fehlerhaft_Fahrleistung.y"]
    )
})

print(t02.shape)
display(t02.head())

(3204104, 7)


,ID_T02,Produktionsdatum,Herstellernummer,Werksnummer,Fehlerhaft,Fehlerhaft_Datum,Fehlerhaft_Fahrleistung
1,2-201-2011-239,2008-11-07,201.0,2011.0,1.0,2010-04-09,38354.158904
2,2-201-2011-304,2008-11-07,201.0,2011.0,0.0,NaN,0.000000
3,2-201-2011-125,2008-11-07,201.0,2011.0,1.0,2010-04-09,38354.158904
4,2-201-2011-55,2008-11-07,201.0,2011.0,0.0,NaN,0.000000
5,2-201-2011-133,2008-11-07,201.0,2011.0,0.0,NaN,0.000000


## 4. Identify Affected T2 Control Units

Two groups of T2 control units are affected according to the case description.

### Group 1

Control units produced by:

- Manufacturer: **202**
- Plant: **2022**
- Production period: **April 2009 to November 2014**

### Group 2

Control units produced by:

- Manufacturer: **201**
- Plant: **2011**
- Production number: **1250 to 19500**

Both groups are identified separately and then combined into one affected T2 dataset.

In [4]:
t02_202 = t02[
    (t02["Herstellernummer"] == 202) &
    (t02["Werksnummer"] == 2022)
].copy()

t02_202["Produktionsdatum"] = pd.to_datetime(
    t02_202["Produktionsdatum"]
)

affected_202 = t02_202[
    (t02_202["Produktionsdatum"] >= "2009-04-01") &
    (t02_202["Produktionsdatum"] < "2014-12-01")
].copy()

print("Affected 202/2022:", affected_202.shape)

Affected 202/2022: (1589783, 7)


In [5]:
t02_201 = t02[
    (t02["Herstellernummer"] == 201) &
    (t02["Werksnummer"] == 2011)
].copy()

t02_201["Produktionsnummer"] = (
    t02_201["ID_T02"]
    .str.rsplit("-", n=1)
    .str[-1]
    .astype(int)
)

affected_201 = t02_201[
    t02_201["Produktionsnummer"].between(1250, 19500)
].copy()

print("Affected 201/2011:", affected_201.shape)

Affected 201/2011: (18251, 8)


In [6]:
affected_t2 = pd.concat(
    [affected_202, affected_201],
    ignore_index=True
)

print("Affected T2 units:", affected_t2.shape)
print("Unique T2 IDs:", affected_t2["ID_T02"].nunique())
print(
    "Duplicate T2 IDs:",
    affected_t2["ID_T02"].duplicated().sum()
)

Affected T2 units: (1608034, 8)
Unique T2 IDs: 1608034
Duplicate T2 IDs: 0


## 5. Trace Affected T2 Units to K1 Engine Components

The affected T2 units are installed in K1 engine components.

Four K1 component datasets are relevant:

- K1BE1
- K1BE2
- K1DI1
- K1DI2

Each table contains an `ID_T2` column and the corresponding K1 component ID.

The affected T2 IDs are therefore used to filter each K1 dataset.

The resulting relationships are combined into one mapping table between affected T2 units and K1 engine components.

In [7]:
affected_t2_ids = set(affected_t2["ID_T02"])

k1be1 = pd.read_csv(
    r".\data\Bestandteile_Komponente_K1BE1.csv",
    sep=";",
    quotechar='"'
)

k1be2 = pd.read_csv(
    r".\data\Bestandteile_Komponente_K1BE2.csv",
    sep=";",
    quotechar='"'
)

k1di1 = pd.read_csv(
    r".\data\Bestandteile_Komponente_K1DI1.csv",
    sep=";",
    quotechar='"'
)

k1di2 = pd.read_csv(
    r".\data\Bestandteile_Komponente_K1DI2.csv",
    sep=";",
    quotechar='"'
)

In [8]:
k1be1_affected = k1be1[
    k1be1["ID_T2"].isin(affected_t2_ids)
][["ID_T2", "ID_K1BE1"]].copy()

k1be2_affected = k1be2[
    k1be2["ID_T2"].isin(affected_t2_ids)
][["ID_T2", "ID_K1BE2"]].copy()

k1di1_affected = k1di1[
    k1di1["ID_T2"].isin(affected_t2_ids)
][["ID_T2", "ID_K1DI1"]].copy()

k1di2_affected = k1di2[
    k1di2["ID_T2"].isin(affected_t2_ids)
][["ID_T2", "ID_K1DI2"]].copy()

In [9]:
k1be1_map = k1be1_affected.copy()
k1be1_map["ID_K1"] = k1be1_map["ID_K1BE1"]
k1be1_map["K1_Type"] = "K1BE1"

k1be2_map = k1be2_affected.rename(
    columns={"ID_K1BE2": "ID_K1"}
)
k1be2_map["K1_Type"] = "K1BE2"

k1di1_map = k1di1_affected.rename(
    columns={"ID_K1DI1": "ID_K1"}
)
k1di1_map["K1_Type"] = "K1DI1"

k1di2_map = k1di2_affected.rename(
    columns={"ID_K1DI2": "ID_K1"}
)
k1di2_map["K1_Type"] = "K1DI2"

affected_t2_k1 = pd.concat(
    [
        k1be1_map[["ID_T2", "ID_K1", "K1_Type"]],
        k1be2_map[["ID_T2", "ID_K1", "K1_Type"]],
        k1di1_map[["ID_T2", "ID_K1", "K1_Type"]],
        k1di2_map[["ID_T2", "ID_K1", "K1_Type"]]
    ],
    ignore_index=True
)

print(affected_t2_k1.shape)
display(affected_t2_k1.head())

(1608034, 3)


,ID_T2,ID_K1,K1_Type
0,2-201-2011-1301,K1BE1-101-1011-151,K1BE1
1,2-201-2011-1271,K1BE1-101-1011-155,K1BE1
2,2-201-2011-1300,K1BE1-104-1041-374,K1BE1
3,2-201-2011-1252,K1BE1-101-1011-252,K1BE1
4,2-201-2011-1323,K1BE1-101-1011-256,K1BE1


In [10]:
print("Rows:", len(affected_t2_k1))
print("Unique T2:", affected_t2_k1["ID_T2"].nunique())
print(
    "Duplicate T2:",
    affected_t2_k1["ID_T2"].duplicated().sum()
)

print(
    affected_t2_k1["K1_Type"].value_counts()
)

Rows: 1608034
Unique T2: 1608034
Duplicate T2: 0
K1_Type
K1DI1    599042
K1BE1    597655
K1BE2    205722
K1DI2    205615
Name: count, dtype: int64


## 6. Identify Affected OEM1 Vehicles

The OEM1 vehicle parts lists contain the engine component installed in each vehicle in the column `ID_Motor`.

Because the affected K1 components are the affected engine components, vehicles are identified by checking whether their `ID_Motor` occurs in the set of affected K1 IDs.

The analysis is performed separately for:

- OEM1 Type11
- OEM1 Type12

The two vehicle datasets are then combined.

In [11]:
vehicles_11 = pd.read_csv(
    r".\data\Bestandteile_Fahrzeuge_OEM1_Typ11.csv",
    sep=";",
    quotechar='"'
)

vehicles_12 = pd.read_csv(
    r".\data\Bestandteile_Fahrzeuge_OEM1_Typ12.csv",
    sep=";",
    quotechar='"'
)

affected_k1_ids = set(affected_t2_k1["ID_K1"])

In [12]:
vehicles_11_affected = vehicles_11[
    vehicles_11["ID_Motor"].isin(affected_k1_ids)
].copy()

vehicles_12_affected = vehicles_12[
    vehicles_12["ID_Motor"].isin(affected_k1_ids)
].copy()

print(
    "Affected Type11 vehicles:",
    vehicles_11_affected.shape
)

print(
    "Affected Type12 vehicles:",
    vehicles_12_affected.shape
)

Affected Type11 vehicles: (992252, 6)
Affected Type12 vehicles: (204445, 6)


In [13]:
print(
    "Type11 rows:",
    len(vehicles_11_affected)
)

print(
    "Unique Type11 vehicles:",
    vehicles_11_affected["ID_Fahrzeug"].nunique()
)

print(
    "Duplicate Type11 vehicle IDs:",
    vehicles_11_affected["ID_Fahrzeug"].duplicated().sum()
)

print(
    "Type12 rows:",
    len(vehicles_12_affected)
)

print(
    "Unique Type12 vehicles:",
    vehicles_12_affected["ID_Fahrzeug"].nunique()
)

print(
    "Duplicate Type12 vehicle IDs:",
    vehicles_12_affected["ID_Fahrzeug"].duplicated().sum()
)

Type11 rows: 992252
Unique Type11 vehicles: 992252
Duplicate Type11 vehicle IDs: 0
Type12 rows: 204445
Unique Type12 vehicles: 204445
Duplicate Type12 vehicle IDs: 0


In [14]:
affected_vehicles_11 = vehicles_11_affected[
    ["ID_Motor", "ID_Fahrzeug"]
].copy()

affected_vehicles_11["Vehicle_Type"] = "Type11"

affected_vehicles_12 = vehicles_12_affected[
    ["ID_Motor", "ID_Fahrzeug"]
].copy()

affected_vehicles_12["Vehicle_Type"] = "Type12"

affected_vehicles = pd.concat(
    [
        affected_vehicles_11,
        affected_vehicles_12
    ],
    ignore_index=True
)

print(affected_vehicles.shape)
display(affected_vehicles.head())

(1196697, 3)


,ID_Motor,ID_Fahrzeug,Vehicle_Type
0,K1BE1-104-1041-536,11-1-11-273,Type11
1,K1BE1-104-1041-760,11-1-11-432,Type11
2,K1BE1-104-1041-515,11-1-11-520,Type11
3,K1BE1-104-1041-374,11-1-11-580,Type11
4,K1BE1-102-1021-97,11-1-11-615,Type11


## 7. Link Affected Vehicles to Registration Data

The registration dataset contains:

- `IDNummer` – vehicle identifier
- `Gemeinden` – municipality
- `Zulassung` – registration date

The vehicle identifier in the production data is `ID_Fahrzeug`.

Therefore, the registration data are linked using:

`ID_Fahrzeug = IDNummer`

The registration table contains unique vehicle identifiers, so a one-to-one merge is expected.

In [15]:
registrations = pd.read_csv(
    r".\data\Zulassungen_alle_Fahrzeuge.csv",
    sep=";",
    quotechar='"'
)

print("Registration rows:", len(registrations))
print(
    "Unique IDNummer:",
    registrations["IDNummer"].nunique()
)

print(
    "Duplicate IDNummer:",
    registrations["IDNummer"].duplicated().sum()
)

Registration rows: 3204104
Unique IDNummer: 3204104
Duplicate IDNummer: 0


In [16]:
registrations_small = registrations[
    ["IDNummer", "Gemeinden", "Zulassung"]
].copy()

affected_registered = affected_vehicles.merge(
    registrations_small,
    left_on="ID_Fahrzeug",
    right_on="IDNummer",
    how="left",
    validate="one_to_one"
)

print(affected_registered.shape)
display(affected_registered.head())

(1196697, 6)


,ID_Motor,ID_Fahrzeug,Vehicle_Type,IDNummer,Gemeinden,Zulassung
0,K1BE1-104-1041-536,11-1-11-273,Type11,11-1-11-273,LUGAU/ERZGEB.,2009-01-02
1,K1BE1-104-1041-760,11-1-11-432,Type11,11-1-11-432,SUEDBROOKMERLAND,2009-01-02
2,K1BE1-104-1041-515,11-1-11-520,Type11,11-1-11-520,HEMMINGEN,2009-01-02
3,K1BE1-104-1041-374,11-1-11-580,Type11,11-1-11-580,HABICHTSWALD,2009-01-02
4,K1BE1-102-1021-97,11-1-11-615,Type11,11-1-11-615,BURGWALD,2009-01-02


## 8. Validate the Registration Merge

The registration merge is checked for vehicles for which no municipality was found.

A successful merge should result in no affected vehicles with a missing municipality.

In [17]:
print(
    "Affected vehicles without registration:",
    affected_registered["Gemeinden"].isna().sum()
)

print(
    "Affected vehicles with registration:",
    affected_registered["Gemeinden"].notna().sum()
)

Affected vehicles without registration: 0
Affected vehicles with registration: 1196697


## 9. Affected Vehicles by Municipality

The final registered vehicle dataset can now be aggregated by municipality.

This provides the number of affected OEM1 vehicles registered in each municipality.

In [18]:
municipality_counts = (
    affected_registered["Gemeinden"]
    .value_counts()
    .rename_axis("Gemeinde")
    .reset_index(name="Affected_Vehicles")
)

print(municipality_counts.head(20))

                Gemeinde  Affected_Vehicles
0                  KOELN              30788
1               DORTMUND              19123
2                LEIPZIG              15474
3                DRESDEN              15360
4                 BOCHUM              13308
5              BIELEFELD              11755
6                   BONN              10270
7              MUENSTER1               9474
8               MUENSTER               9236
9               AUGSBURG               9103
10         GELSENKIRCHEN               8914
11          BRAUNSCHWEIG               8644
12                AACHEN               7530
13                 MAINZ               6440
14            LEVERKUSEN               6024
15  FREIBURG IM BREISGAU               5563
16                 HERNE               5371
17            OSNABRUECK               5330
18               BOTTROP               4837
19            REGENSBURG               4749


In [19]:
print("Number of municipalities:", len(municipality_counts))

Number of municipalities: 5627
